### Analysis - Continuous Tracking

In [46]:
import numpy as np
from scipy.signal import correlate, correlation_lags
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict
import pandas as pd

#### Cross Correlogram

In [47]:

def compute_cross_correlogram(target_signal, cursor_signal, sampling_rate, max_lag_seconds) -> tuple:
    """Compute cross-correlogram and extract metrics between two signals.
    Parameters:
        target_signal (np.ndarray): The reference signal (e.g., target movement).
        cursor_signal (np.ndarray): The signal to compare against the target (e.g., cursor movement).
        sampling_rate (float): Sampling rate of the signals in Hz.
        max_lag_seconds (float): Maximum lag to consider for the cross-correlogram in seconds.
    Returns:
        lags (np.ndarray): Array of lag times in seconds.
        correlation_scores (np.ndarray): Cross-correlation values at each lag.
        optimal_lag_seconds (float): Lag time at which the correlation is maximal.
        peak_correlation (float): Maximum correlation value.
        width (float): Full Width at Half Maximum (FWHM) of the correlation peak.
        # area (float): Area under the cross-correlogram curve within the max lag window.
    """
    # STEP 1: Center the signals
    target_centered = target_signal - np.mean(target_signal)
    cursor_centered = cursor_signal - np.mean(cursor_signal)
    
    # STEP 2 & 3: Compute raw cross-correlation using SciPy
    # This automatically computes all possible lags
    raw_corr = correlate(target_centered, cursor_centered, mode='full')
    all_lags = correlation_lags(len(target_centered), len(cursor_centered), mode='full')
    
    # Crop to your specified max_lag_seconds window
    max_lag_frames = int(max_lag_seconds * sampling_rate)
    window_mask = (all_lags >= -max_lag_frames) & (all_lags <= max_lag_frames)
    
    lags_frames = all_lags[window_mask]
    raw_corr_window = raw_corr[window_mask]
    lags = lags_frames / sampling_rate
    
    # STEP 4: Proper Pearson Correlation Normalization per lag
    # We must calculate standard deviations based on the exact overlapping slices
    correlation_scores = np.zeros_like(raw_corr_window, dtype=float)
    N = len(target_signal)
    
    for i, lag in enumerate(lags_frames):
        if lag < 0:
            t_slice = target_centered[-lag:]
            c_slice = cursor_centered[:N + lag]
        else:
            t_slice = target_centered[:N - lag]
            c_slice = cursor_centered[lag:]
            
        denom = np.std(t_slice) * np.std(c_slice) * len(t_slice)
        if denom > 0:
            correlation_scores[i] = np.sum(t_slice * c_slice) / denom

    # STEP 5: Extract Metrics safely
    peak_index = np.argmax(correlation_scores)
    peak_correlation = correlation_scores[peak_index]
    optimal_lag_seconds = lags[peak_index]
    
    # Calculate Width (FWHM) properly relative to the baseline/peak floor
    half_max_threshold = peak_correlation / 2.0
    
    # Find where the curve drops below half-max on the left and right sides of the peak
    left_side = correlation_scores[:peak_index]
    right_side = correlation_scores[peak_index:]
    
    # Find closest crossing points
    left_idx = np.where(left_side < half_max_threshold)[0]
    right_idx = np.where(right_side < half_max_threshold)[0]
    
    idx_start = left_idx[-1] if len(left_idx) > 0 else 0
    idx_end = peak_index + right_idx[0] if len(right_idx) > 0 else len(correlation_scores) - 1
    
    width = lags[idx_end] - lags[idx_start]
    
    # Area under the curve (using Trapezoidal rule within your max lag window)
    # area = np.trapz(correlation_scores, lags)
    
    return lags, correlation_scores, optimal_lag_seconds, peak_correlation, width, # area


In [48]:


def plot_cross_correlogram(lags, correlation_scores, optimal_lag_seconds, peak_correlation, width) -> None:
    """Helper function to plot the cross-correlogram with key metrics highlighted.

    Parameters:
    - lags: Array of lag times (in seconds).
    - correlation_scores: Array of correlation coefficients corresponding to each lag.
    - optimal_lag_seconds: The lag time (in seconds) at which the correlation is maximized.
    - peak_correlation: The maximum correlation value.
    - width: The full width at half maximum (FWHM) of the peak.

    Returns:
    - None
    """
    
    plt.figure(figsize=(10, 6))
    plt.plot(lags, correlation_scores, label='Cross-Correlogram', color='blue')
    plt.axvline(optimal_lag_seconds, color='red', linestyle='--', label=f'Optimal Lag: {optimal_lag_seconds:.3f}s')
    plt.axhline(peak_correlation / 2, color='green', linestyle='--', label=f'Half-Max: {peak_correlation / 2:.3f}')
    
    # Highlight the width (FWHM)
    plt.fill_between(lags, 0, correlation_scores, where=(lags >= optimal_lag_seconds - width/2) & (lags <= optimal_lag_seconds + width/2), 
                     color='orange', alpha=0.5, label=f'Width (FWHM): {width:.3f}s')
    
    plt.title('Cross-Correlogram between Target and Cursor Signals')
    plt.xlabel('Lag (seconds)')
    plt.ylabel('Correlation Coefficient')
    plt.legend()
    plt.grid()
    plt.show()

#### Position error

In [49]:
def compute_position_error(target_signal, cursor_signal) -> float:
    """Compute the mean Euclidean distance between the target and cursor signals.
    
    Parameters:
    target_signal (np.ndarray): The target signal array.
    cursor_signal (np.ndarray): The cursor signal array.

    Returns:
    float: The mean Euclidean distance between the target and cursor signals.
    """
    return np.mean(np.sqrt((target_signal - cursor_signal) ** 2))

#### Velocity Error (eye & cursor)

In [50]:
def compute_velocity_error(target_signal, cursor_signal, sampling_rate) -> float:
    """Compute the mean absolute difference in velocity between the target and cursor signals.
    
    Parameters:
    target_signal (np.ndarray): The target signal array.
    cursor_signal (np.ndarray): The cursor signal array.
    sampling_rate (float): The sampling rate of the signals in Hz.

    Returns:
    float: The mean absolute velocity error between the target and cursor signals.
    """
    # Compute velocity by taking the first derivative (difference) of the signals
    target_velocity = np.diff(target_signal) * sampling_rate
    cursor_velocity = np.diff(cursor_signal) * sampling_rate
    
    # Compute mean absolute velocity error
    return np.mean(np.abs(target_velocity - cursor_velocity))

#### Position Variance (eye & cursor)

In [51]:
def compute_position_variance(signal: np.ndarray) -> float:
    """Compute the variance of the position signal to measure how much it spreads out or drifts around its own average trajectory.
    
    Parameters:
    signal (np.ndarray): The position signal array.

    Returns:
    float: The variance of the position signal.
    """
    return np.var(signal)

#### Saccade rate (eye)

In [52]:
def compute_saccade_rate(signal: np.ndarray, sampling_rate: float, velocity_threshold: float) -> float:
    """Compute the saccade rate using eyelinks saccade detection method based on velocity thresholding.
    
    Parameters:
    signal (np.ndarray): The position signal array.
    sampling_rate (float): The sampling rate of the signal in Hz.
    velocity_threshold (float): The velocity threshold to identify saccades.

    Returns:
    float: The saccade rate in saccades per second.
    """
    # Compute velocity by taking the first derivative (difference) of the signal
    velocity = np.diff(signal) * sampling_rate
    
    # Count the number of saccades based on the velocity threshold
    saccades = np.sum(np.abs(velocity) > velocity_threshold)
    
    # Calculate saccade rate (saccades per second)
    duration_seconds = len(signal) / sampling_rate
    return saccades / duration_seconds if duration_seconds > 0 else 0.0


#### Gain (eye)

In [53]:
def compute_gain_eye(target_signal, cursor_signal) -> float:
    """Compute the gain of the eye movement by comparing the amplitude of the cursor signal to the target signal.
    
    Parameters:
    target_signal (np.ndarray): The target signal array.
    cursor_signal (np.ndarray): The cursor signal array.

    Returns:
    float: The gain of the eye movement.
    """
    # Compute the amplitude (peak-to-peak) of both signals
    target_amplitude = np.ptp(target_signal)
    cursor_amplitude = np.ptp(cursor_signal)
    
    # Compute gain as the ratio of cursor amplitude to target amplitude
    return cursor_amplitude / target_amplitude if target_amplitude > 0 else 0.0

### compute all metrics for each participant

In [54]:
def compute_one_trial(
    trial_data: pd.DataFrame,
    sampling_rate,
    max_lag_seconds,
    velocity_threshold,
    target_signal_col: str = "target_pos_x",
    cursor_signal_col: str = "cursor_pos_x",
) -> dict:
    """Compute all relevant metrics for one trial dataframe.
    
    Parameters:
    trial_data (pd.DataFrame): The rows for a single trial.
    sampling_rate (float): The sampling rate of the signals in Hz.
    max_lag_seconds (float): Maximum lag to consider for the cross-correlogram in seconds.
    velocity_threshold (float): The velocity threshold to identify saccades.

    Returns:
    dict: A dictionary containing all computed metrics for the trial.
    """
    required_columns = {target_signal_col, cursor_signal_col}
    missing_columns = required_columns.difference(trial_data.columns)
    if missing_columns:
        raise KeyError(f"Missing required columns for trial metrics: {sorted(missing_columns)}")

    cleaned_trial = trial_data[[target_signal_col, cursor_signal_col]].dropna()
    if cleaned_trial.empty:
        return {}

    target_signal = cleaned_trial[target_signal_col].to_numpy()
    cursor_signal = cleaned_trial[cursor_signal_col].to_numpy()

    lags, correlation_scores, optimal_lag_seconds, peak_correlation, width = compute_cross_correlogram(
        target_signal, cursor_signal, sampling_rate, max_lag_seconds
    )
    
    position_error = compute_position_error(target_signal, cursor_signal)
    velocity_error = compute_velocity_error(target_signal, cursor_signal, sampling_rate)
    position_variance = compute_position_variance(cursor_signal)
    saccade_rate = compute_saccade_rate(cursor_signal, sampling_rate, velocity_threshold)
    gain_eye = compute_gain_eye(target_signal, cursor_signal)
    
    trial_metrics = {
        'lags': lags,
        'correlation_scores': correlation_scores,
        'optimal_lag_seconds': optimal_lag_seconds,
        'peak_correlation': peak_correlation,
        'width': width,
        'position_error': position_error,
        'velocity_error': velocity_error,
        'position_variance': position_variance,
        'saccade_rate': saccade_rate,
        'gain_eye': gain_eye,
    }

    for metadata_column in ('trial_number', 'training', 'target_width', 'state_marker'):
        if metadata_column in trial_data.columns:
            value = trial_data[metadata_column].iloc[0]
            if pd.notna(value):
                trial_metrics[metadata_column] = bool(value) if metadata_column == 'training' else (
                    value.item() if hasattr(value, 'item') else value
                )

    return trial_metrics

In [55]:
def compute_one_block(
    block_data: pd.DataFrame,
    sampling_rate,
    max_lag_seconds,
    velocity_threshold,
    target_signal_col: str = "target_pos_x",
    cursor_signal_col: str = "cursor_pos_x",
) -> pd.DataFrame:
    """Compute metrics for all trials in a block and return a DataFrame with the results.
    
    Parameters:
    block_data (pd.DataFrame): All rows belonging to one block.
    sampling_rate (float): The sampling rate of the signals in Hz.
    max_lag_seconds (float): Maximum lag to consider for the cross-correlogram in seconds.
    velocity_threshold (float): The velocity threshold to identify saccades.

    Returns:
    pd.DataFrame: A DataFrame containing computed metrics for all trials in the block.
    """
    results = []

    for trial_number, trial_data in block_data.groupby('trial_number', sort=True):
        trial_metrics = compute_one_trial(
            trial_data,
            sampling_rate,
            max_lag_seconds,
            velocity_threshold,
            target_signal_col=target_signal_col,
            cursor_signal_col=cursor_signal_col,
        )
        if not trial_metrics:
            continue
        trial_metrics['trial_number'] = int(trial_number)
        if 'training' in block_data.columns:
            trial_metrics['training'] = bool(block_data['training'].iloc[0])
        if 'target_width' in block_data.columns and pd.notna(block_data['target_width'].iloc[0]):
            trial_metrics['target_width'] = float(block_data['target_width'].iloc[0])
        results.append(trial_metrics)

    return pd.DataFrame(results)

In [56]:
TARGET_WIDTH_BY_BLOCK = {
    0: 0.15,
    1: 0.30,
    2: 0.45,
    3: 0.60,
}

UNCERTAINTY_LEVEL_BY_BLOCK = {
    0: 0.2,
    1: 0.3,
    2: 0.4,
    3: 0.5,
}

BLOCK_BY_TARGET_WIDTH = {width: block for block, width in TARGET_WIDTH_BY_BLOCK.items()}


def compute_one_participant(
    participant_data: pd.DataFrame,
    participant_id: str,
    sampling_rate,
    max_lag_seconds,
    velocity_threshold,
    target_signal_col: str = "target_pos_x",
    cursor_signal_col: str = "cursor_pos_x",
    include_training: bool = False,
) -> pd.DataFrame:
    """Compute one aggregated row per target width for a participant.
    
    Parameters:
    participant_data (pd.DataFrame): All rows for one participant.
    participant_id (str): Identifier for the participant folder.
    sampling_rate (float): The sampling rate of the signals in Hz.
    max_lag_seconds (float): Maximum lag to consider for the cross-correlogram in seconds.
    velocity_threshold (float): The velocity threshold to identify saccades.

    Returns:
    pd.DataFrame: A DataFrame containing one row per target width with aggregated metrics.
    """
    if participant_data.empty:
        return pd.DataFrame()

    participant_trials = participant_data.copy()
    if not include_training and 'training' in participant_trials.columns:
        participant_trials = participant_trials.loc[~participant_trials['training']].copy()

    if 'target_width' not in participant_trials.columns:
        raise KeyError("participant_data must contain a 'target_width' column.")

    all_results = []
    metric_columns = [
        'optimal_lag_seconds',
        'peak_correlation',
        'width',
        'position_error',
        'velocity_error',
        'position_variance',
    ]

    for target_width, block_data in participant_trials.groupby('target_width', sort=True):
        block_metrics = compute_one_block(
            block_data,
            sampling_rate,
            max_lag_seconds,
            velocity_threshold,
            target_signal_col=target_signal_col,
            cursor_signal_col=cursor_signal_col,
        )
        if block_metrics.empty:
            continue

        summary = block_metrics[metric_columns].mean(numeric_only=True)
        all_results.append({
            'participant_id': participant_id,
            'target_width': float(target_width),
            'lag': summary['optimal_lag_seconds'],
            'peak': summary['peak_correlation'],
            'width': summary['width'],
            'position_error': summary['position_error'],
            'velocity_error': summary['velocity_error'],
            'position_variance': summary['position_variance'],
        })

    if not all_results:
        return pd.DataFrame()

    return pd.DataFrame(all_results, columns=['participant_id', 'target_width', 'lag', 'peak', 'width', 'position_error', 'velocity_error', 'position_variance'])

In [57]:
def compute_all_participants(
    data_root: Path,
    sampling_rate,
    max_lag_seconds,
    velocity_threshold,
    target_signal_col: str = "target_pos_x",
    cursor_signal_col: str = "cursor_pos_x",
    include_training: bool = False,
) -> pd.DataFrame:
    """Compute metrics for all participants in the continuous data folder.
    
    Parameters:
    data_root (Path): The root folder containing one subfolder per participant.
    sampling_rate (float): The sampling rate of the signals in Hz.
    max_lag_seconds (float): Maximum lag to consider for the cross-correlogram in seconds.
    velocity_threshold (float): The velocity threshold to identify saccades.

    Returns:
    pd.DataFrame: A DataFrame containing computed metrics for all participants.
    """
    all_results = []

    for participant_dir in sorted(path for path in data_root.iterdir() if path.is_dir()):
        csv_files = sorted(participant_dir.glob('*.csv'))
        if not csv_files:
            continue

        participant_frames = [pd.read_csv(csv_file) for csv_file in csv_files]
        participant_data = pd.concat(participant_frames, ignore_index=True)
        participant_metrics = compute_one_participant(
            participant_data,
            participant_dir.name,
            sampling_rate,
            max_lag_seconds,
            velocity_threshold,
            target_signal_col=target_signal_col,
            cursor_signal_col=cursor_signal_col,
            include_training=include_training,
        )
        if not participant_metrics.empty:
            all_results.append(participant_metrics)

    if not all_results:
        return pd.DataFrame()

    return pd.concat(all_results, ignore_index=True)

#### load data

In [58]:
continuous_data_path = Path(r'c:\Users\HP\Documents\uni\SoSe26\expra\code\analysis\data\continuous')
sampling_rate = 144
max_lag_seconds = 1.0
velocity_threshold = 50.0
continuous_metrics = compute_all_participants(
    continuous_data_path,
    sampling_rate,
    max_lag_seconds,
    velocity_threshold,
)
continuous_metrics

,participant_id,target_width,lag,peak,width,position_error,velocity_error,position_variance
0,ch51,0.15,0.422222,0.973919,1.941667,0.258505,4.785041,0.643520
1,ch51,0.30,0.494444,0.962783,1.926389,0.267967,4.737831,0.607735
2,ch51,0.45,0.478472,0.987421,2.000000,0.284295,4.747524,1.800220
3,ch51,0.60,0.600694,0.956058,1.916667,0.311021,4.725833,0.635859
4,ir31,0.15,0.317361,0.976632,1.898611,0.221561,4.763471,0.590171
5,ir31,0.30,0.406944,0.964420,1.924306,0.259452,4.753816,0.826251
6,ir31,0.45,0.769444,0.928729,1.831944,0.356503,4.755829,0.704884
7,ir31,0.60,0.728472,0.893195,1.856250,0.351586,4.702389,0.477183
8,lj25,0.15,0.425694,0.980793,1.970833,0.254833,4.739459,0.723651
9,lj25,0.30,0.522917,0.966756,1.916667,0.268644,4.745554,0.774952
